# 01 — Raw EEG Exploration

Phase 1 notebook: load one KaraOne subject, verify shape/labels, visualize signal.

**Stop condition:** `(n_epochs, n_eeg_channels, n_times)` array with correct labels before moving to preprocessing.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import mne
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for inline plots
import matplotlib.pyplot as plt

from src.loader import load_subject, list_available_subjects, SUBJECTS

mne.set_log_level('WARNING')
print('MNE version:', mne.__version__)

In [ ]:
# Check what's available on disk
available = list_available_subjects('../data/raw')
print(f'Available subjects ({len(available)}/{len(SUBJECTS)}): {available}')

In [ ]:
SUBJECT = available[0] if available else 'MM05'
print(f'Loading subject: {SUBJECT}')

epochs, labels = load_subject(
    SUBJECT,
    raw_data_dir='../data/raw',
    epoch_type='thinking',  # imagined speech condition
    verbose=True
)

In [ ]:
# === SHAPE VERIFICATION ===
data = epochs.get_data()
print(f'Epochs shape: {data.shape}  (expected: n_trials × n_eeg_channels × n_times)')
print(f'  n_trials:       {data.shape[0]}')
print(f'  n_eeg_channels: {data.shape[1]}')
print(f'  n_times:        {data.shape[2]}  ({data.shape[2] / epochs.info["sfreq"]:.1f}s @ {epochs.info["sfreq"]}Hz)')
print(f'Sampling rate: {epochs.info["sfreq"]} Hz')
print(f'Channel names (first 10): {epochs.ch_names[:10]}')
print(f'Total channels: {len(epochs.ch_names)}')

In [ ]:
# === LABEL VERIFICATION ===
unique, counts = np.unique(labels, return_counts=True)
print(f'\nLabel distribution ({len(labels)} total trials):')
for word, count in sorted(zip(unique, counts)):
    print(f'  {word:8s}: {count} trials')

print(f'\nExpected: 11 classes (pat, pot, knew, gnaw + 7 phonemes)')
print(f'Found: {len(unique)} unique labels')

# Sanity check: labels should be from the known set
from src.loader import EVENT_ID
unknown = set(unique) - set(EVENT_ID.keys())
if unknown:
    print(f'WARNING: Unknown labels: {unknown}')
else:
    print('All labels valid ✓')

In [ ]:
# === SIGNAL AMPLITUDE CHECK ===
print(f'Signal range: {data.min():.1f} to {data.max():.1f} µV')
print(f'Mean amplitude: {np.abs(data).mean():.2f} µV')
print()
# EEG should be ±100 µV range; larger = bad channels or artifact
if np.abs(data).max() > 500:
    print('WARNING: Amplitude > 500 µV — check for bad channels')
else:
    print('Amplitude range looks reasonable ✓')

In [ ]:
# === RAW TRACE PLOT — first 5 channels, first trial ===
fig, axes = plt.subplots(5, 1, figsize=(14, 8), sharex=True)
times = np.linspace(0, data.shape[2] / epochs.info['sfreq'], data.shape[2])

for i, ax in enumerate(axes):
    ax.plot(times, data[0, i, :], lw=0.5, color='royalblue')
    ax.set_ylabel(epochs.ch_names[i], fontsize=8)
    ax.set_ylim(-150, 150)
    ax.axhline(0, color='gray', lw=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'{SUBJECT} — Trial 0 ({labels[0]}) — First 5 EEG channels', fontsize=11)
plt.tight_layout()
plt.savefig(f'../outputs/figures/{SUBJECT}_raw_traces.png', dpi=120, bbox_inches='tight')
plt.show()
print('Raw trace plot saved.')

In [ ]:
# === POWER SPECTRAL DENSITY ===
# Compute PSD on all epochs, average across trials
spectrum = epochs.compute_psd(method='welch', fmin=0.5, fmax=100, n_fft=2048, n_overlap=512)
psd, freqs = spectrum.get_data(return_freqs=True)
# Average across trials and channels
mean_psd = psd.mean(axis=(0, 1))  # (n_trials, n_channels, n_freqs) → (n_freqs,)

fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(freqs, mean_psd, lw=1.2, color='steelblue')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (µV²/Hz)')
ax.set_title(f'{SUBJECT} — Mean PSD across all thinking epochs')
ax.set_xlim(0, 100)

# Mark frequency bands
band_colors = {'Delta\n0-4': (0,4,'#e8d5b7'), 'Theta\n4-8': (4,8,'#b7d5e8'),
               'Alpha\n8-12': (8,12,'#b7e8c8'), 'Beta\n12-30': (12,30,'#e8b7d5'),
               'Gamma\n30+': (30,60,'#d5e8b7')}
for name, (lo, hi, color) in band_colors.items():
    ax.axvspan(lo, hi, alpha=0.15, color=color, label=name)

# 60 Hz line noise marker
ax.axvline(60, color='red', lw=1, ls='--', alpha=0.5, label='60 Hz')
ax.legend(fontsize=7, ncol=6, loc='upper right')

plt.tight_layout()
plt.savefig(f'../outputs/figures/{SUBJECT}_psd.png', dpi=120, bbox_inches='tight')
plt.show()
print('PSD plot saved.')
print()
# Check for 60 Hz spike
idx_60 = np.argmin(np.abs(freqs - 60))
idx_55 = np.argmin(np.abs(freqs - 55))
power_ratio_60 = mean_psd[idx_60] / mean_psd[idx_55]
if power_ratio_60 > 3:
    print(f'60 Hz spike detected (ratio vs 55Hz: {power_ratio_60:.1f}x) — notch filter needed')
else:
    print(f'No strong 60 Hz spike (ratio: {power_ratio_60:.1f}x)')

In [ ]:
# === PER-CLASS TRIAL COUNT ===
print('Trial counts by class:')
for word in sorted(EVENT_ID.keys()):
    count = np.sum(labels == word)
    bar = '#' * count
    print(f'  {word:8s}: {count:3d} {bar}')

# Check minimum trials per class (need ≥5 for 5-fold CV)
min_count = min(np.sum(labels == w) for w in unique)
print(f'\nMin trials per class: {min_count}')
if min_count < 5:
    print('WARNING: Some classes have <5 trials — may need to adjust CV strategy')
else:
    print('Sufficient trials for 5-fold CV ✓')

In [ ]:
# === SUMMARY ===
print('='*50)
print('PHASE 1 VERIFICATION CHECKLIST')
print('='*50)
shape_ok = data.shape[1] >= 60  # should have ~64 EEG channels
labels_ok = len(unique) >= 9  # expect 11 classes
amplitude_ok = np.abs(data).max() < 1000
print(f'[{"OK" if shape_ok else "FAIL"}] EEG channel count: {data.shape[1]} (expected ~64)')
print(f'[{"OK" if labels_ok else "FAIL"}] Label count: {len(unique)} unique (expected 11)')
print(f'[{"OK" if amplitude_ok else "FAIL"}] Amplitude in range: max {np.abs(data).max():.0f} µV')
print(f'[OK] Trial duration: {data.shape[2]/epochs.info["sfreq"]:.1f}s (expected 4.9s)')
print()
if shape_ok and labels_ok and amplitude_ok:
    print('Phase 1 complete — ready for preprocessing (Phase 2)')
else:
    print('Fix issues above before proceeding')